# Week 29, Day 4: Conductances, driving force and shunting inhibition

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w029/d4.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

### Exercise: How linear is a potassium current?

`ghk_K(V)` is the GHK potassium current for 140 mM inside and 5 mM outside. Find its slope at the reversal potential, `g_rev`, by a central difference, and compare it with the linear current g_rev (V - E_K) at 50 mV either side of E_K. Store the ratio of the outward current at E_K + 50 to the size of the inward current at E_K - 50 in `ratio`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
VT = 26.725
E_K = VT * np.log(5 / 140)

def ghk_K(V):
    u = V / VT
    return u * (140 - 5 * np.exp(-u)) / (1 - np.exp(-u))

g_rev = ...
ratio = ...
for V in (E_K - 50, E_K + 50):
    print(f"V = {V:6.1f} mV: GHK {ghk_K(V):+6.1f}, linear {g_rev * (V - E_K):+6.1f}")
print(f"outward / inward at equal driving force: {ratio:.2f}")
V = np.linspace(-140, 40, 200); V = V[np.abs(V) > 0.01]
plt.plot(V, ghk_K(V), label="GHK"); plt.plot(V, g_rev * (V - E_K), "--", label="linear")
plt.axhline(0, color="gray", lw=0.5); plt.xlabel("V (mV)"); plt.ylabel("current (arbitrary units)"); plt.legend(); plt.show()

In [ ]:
import numpy as np
def test_slope():
    "g_rev is the slope of the GHK current at E_K, by central differences"
    assert abs(g_rev - (ghk_K(E_K + 1e-3) - ghk_K(E_K - 1e-3)) / 2e-3) < 1e-6

def test_ratio():
    "the GHK current 50 mV above E_K is more than twice the size of the current 50 mV below"
    assert abs(ratio - (-ghk_K(E_K + 50) / ghk_K(E_K - 50))) < 1e-9 and ratio > 2

test_slope()
test_ratio()
print("All checks passed")

### Exercise: An EPSP with and without a shunt

With g_L = 10 nS, E_L = -70 mV and C = 200 pF, write `v_inf(gL, ge, gi)` with E_e = 0 and E_i = -70 mV. For an excitatory conductance of 1 nS, compute the steady depolarisation `epsp[gi]` and effective time constant `tau_eff[gi]` (ms) for inhibitory conductances of 0, 5 and 10 nS.

In [ ]:
import numpy as np
E_L, E_e, E_i, C = -70.0, 0.0, -70.0, 200.0         # mV, mV, mV, pF

def v_inf(gL, ge, gi):
    return ...

epsp, tau_eff = {}, {}
for gi in (0, 5, 10):
    epsp[gi] = ...
    tau_eff[gi] = ...
    print(f"g_i = {gi:2d} nS: depolarisation {epsp[gi]:.2f} mV, effective time constant {tau_eff[gi]:.1f} ms;  alone it moves V by {v_inf(10, 0, gi) - E_L:.1f} mV")

In [ ]:
import numpy as np
def test_vinf():
    "v_inf gives the conductance-weighted mean of the reversal potentials"
    assert abs(v_inf(10, 1, 0) - (10 * -70 + 1 * 0) / 11) < 1e-12
    assert abs(v_inf(10, 1, 10) - (10 * -70 + 1 * 0 + 10 * -70) / 21) < 1e-12

def test_shunt():
    "the shunt cuts the depolarisation roughly in half and halves the effective time constant"
    assert 0.45 < epsp[10] / epsp[0] < 0.6 and abs(tau_eff[10] - 200 / 21) < 1e-9

test_vinf()
test_shunt()
print("All checks passed")

### Exercise: Divide or subtract?

Write `rate(I, gi)`: the firing rate of a LIF neuron (C = 200 pF, g_L = 10 nS, E_L = E_i = -70 mV, threshold -50 mV, reset -65 mV, refractory period 2 ms) with total leak g = g_L + g_i, using the exact formula 1 / (t_ref + τ ln((V∞ - V_reset) / (V∞ - V_th))) where V∞ exceeds threshold, else 0. Then find each curve's rheobase (smallest current that fires) and its slope (Hz per nA) from a straight-line fit between 1 and 1.5 nA.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
C, gL, EL, Vth, Vr, tref = 200e-12, 10e-9, -70e-3, -50e-3, -65e-3, 2e-3
I = np.linspace(0, 1.5e-9, 301)

def rate(I, gi):
    g = gL + gi
    vinf, tau = EL + I / g, C / g
    out = np.zeros_like(I)
    fires = vinf > Vth
    out[fires] = ...
    return out

rheobase, slope = {}, {}
high = I >= 1e-9
for gi in (0, 10e-9, 20e-9):
    f = rate(I, gi)
    rheobase[gi] = I[np.argmax(f > 0)]
    slope[gi] = ...
    print(f"g_i = {gi * 1e9:2.0f} nS: rheobase {rheobase[gi] * 1e12:4.0f} pA, slope {slope[gi]:5.1f} Hz/nA, rate at 1 nA {f[200]:5.1f} Hz")
    plt.plot(I * 1e9, f, label=f"g_i = {gi * 1e9:.0f} nS")
plt.xlabel("current (nA)"); plt.ylabel("rate (Hz)"); plt.legend(); plt.show()

In [ ]:
import numpy as np
def test_rate():
    "rate gives the exact LIF firing rate with total leak g_L + g_i and a 2 ms refractory period"
    g = 10e-9 + 10e-9; vinf = -70e-3 + 1e-9 / g
    assert abs(rate(np.array([1e-9]), 10e-9)[0] - 1 / (2e-3 + 200e-12 / g * np.log((vinf + 65e-3) / (vinf + 50e-3)))) < 1e-9

def test_shift():
    "the shunt raises the rheobase by g_i times 20 mV, but the slope at high current falls by less than 20%"
    assert abs(rheobase[10e-9] - rheobase[0] - 10e-9 * 0.02) < 1e-11
    assert slope[10e-9] > 0.8 * slope[0] and slope[20e-9] > 0.8 * slope[0]

test_rate()
test_shift()
print("All checks passed")

### Exercise: A faster neuron in an active network

With C = 200 pF and g_L = 10 nS, background synaptic conductances of 0, 1, 2 and 4 times the leak (`factor`) are added. Compute the effective time constant `tau_eff` (ms) and the time for a synaptic potential to decay to 10% of its peak, `decay_to_10pct` (ms), as τ_eff ln 10.

In [ ]:
import numpy as np
C, gL = 200.0, 10.0                         # pF, nS
factor = [0, 1, 2, 4]
tau_eff = ...
decay_to_10pct = ...
for f, t, w in zip(factor, tau_eff, decay_to_10pct):
    print(f"synaptic conductance {f} x leak: tau_eff = {t:4.1f} ms, a PSP decays to 10% in {w:4.1f} ms")

In [ ]:
import numpy as np
def test_tau():
    "tau_eff divides C by the leak plus the synaptic conductance"
    assert np.allclose(tau_eff, 200 / (10 * (1 + np.array(factor))))

def test_window():
    "decay_to_10pct is tau_eff times ln 10, and falls below 15 ms at four times the leak"
    assert np.allclose(decay_to_10pct, tau_eff * np.log(10)) and decay_to_10pct[-1] < 15

test_tau()
test_window()
print("All checks passed")